## Running open weight model on reasoning prompts

### Model : DeepSeek R1 Distill Qwen 1.5B

### Installing dependencies

In [1]:
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 45.5 MB/s eta 0:00:00


### Imports

In [2]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

### GPU is required to load model fast

#### colab T4 gpu is enough

In [3]:
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Please enable GPU in Colab.")

GPU available: True
GPU name: Tesla T4


### Loading model

In [4]:
model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype = torch.float16,
    device_map = "auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

### Inference function

In [18]:
def ask(prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=1000,
        temperature=0.2,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    answer = full_output[len(formatted_prompt):].strip()

    print(answer)

### Reasoning prompts

In [11]:
prompt1 = "All roses are flowers. Some flowers fade quickly. Do all roses fade quickly?"
prompt2 = "A doctor gives you 3 pills and says take one every half hour. How long before all pills are taken?"
prompt3 = "Every time Rohan carries an umbrella, it rains. Should Rohan stop carrying an umbrella to prevent rain?"
prompt4 = "Aisha is older than Ben. Ben is older than Carlos. Carlos is older than Diana. Who should get the senior discount if it's given to the second youngest?"
prompt5 = "A disease affects 1 in 1000 people. A test for it is 99% accurate. You test positive. What is the probability you actually have the disease?"

### Prompt 1

#### Q. All roses are flowers. Some flowers fade quickly. Do all roses fade quickly?

#### A. No

In [13]:
ask(prompt1)

o figure out whether all roses fade quickly based on the statements "All roses are flowers" and "Some flowers fade quickly." Let me break this down step by step.

First, the statement "All roses are flowers" tells me that roses are a subset of flowers. So, every rose is a flower, but not all flowers are roses. That means if I know something about roses, it's also true about flowers, but the reverse isn't necessarily true.

Next, the statement "Some flowers fade quickly" is a bit more general. It doesn't specify which flowers, just that there's a group of flowers that fade quickly. So, it's possible that roses are part of this group, but it's also possible that they aren't. I don't have enough information to confirm whether roses specifically fade quickly or if they're just another type of flower that happens to fade quickly.

I think the key here is understanding the logic behind these statements. The first one is a universal statement, meaning it applies to all roses. The second is an

### Prompt2

#### Q. A doctor gives you 3 pills and says take one every half hour. How long before all pills are taken?

#### A. 1 hour

In [14]:
ask(prompt2)

rstand the problem. The doctor has given me 3 pills, and I should take one every half hour.

I'll start by noting the time when the first pill is taken, which is at 12:00 PM.

Next, I'll calculate the time for the second pill. Since I take one every half hour, the second pill will be taken 30 minutes after the first one. That brings the time to 12:30 PM.

Finally, I'll determine the time for the third pill. It will be taken another 30 minutes after the second pill, which is at 1:00 PM.

Therefore, all three pills will be taken by 1:00 PM.
</think>

To determine how long it will take to take all three pills, let's break down the process step by step.

1. **First Pill:**
   - **Time Taken:** 0 minutes (at 12:00 PM)

2. **Second Pill:**
   - **Time Taken:** 30 minutes (after the first pill)
   - **Final Time:** 12:00 PM + 30 minutes = **12:30 PM**

3. **Third Pill:**
   - **Time Taken:** 30 minutes (after the second pill)
   - **Final Time:** 12:30 PM + 30 minutes = **1:00 PM**

**Conclus

### Prompt 3

#### Q. Every time Rohan carries an umbrella, it rains. Should Rohan stop carrying an umbrella to prevent rain?

#### A. No

In [15]:
ask(prompt3)

o figure out whether Rohan should stop carrying an umbrella because every time he does, it rains. Hmm, that's an interesting question. Let me break it down.

First, I guess I need to understand the situation. Rohan is carrying an umbrella every time it rains, and that leads to rain. So, does that mean that whenever he carries an umbrella, it rains? Or is it that he's been carrying an umbrella every time it rains, and that's why it's raining? I think it's the latter. So, he's been carrying an umbrella every time it rains, and because he's always carrying it, it's causing the rain.

Now, the question is, should he stop carrying an umbrella to prevent rain? That seems counterintuitive because if he's always carrying it, it's causing the rain. But maybe there's more to it. Let me think about possible reasons why he's carrying the umbrella every time it rains.

Maybe he's trying to stay dry, but if he's always carrying it, he's not getting wet. But the question is about preventing rain, not

### Prompt 4

#### Q. Aisha is older than Ben. Ben is older than Carlos. Carlos is older than Diana. Who should get the senior discount if it's given to the second youngest?

#### A. Carlos

In [16]:
ask(prompt4)

rmine the ages of Aisha, Ben, Carlos, and Diana based on the given information.

From the statements:
- Aisha is older than Ben.
- Ben is older than Carlos.
- Carlos is older than Diana.

This establishes the order of their ages as Aisha > Ben > Carlos > Diana.

Next, I need to identify the second youngest person among them. Since Aisha is the oldest, Ben is the second oldest, Carlos is the third oldest, and Diana is the youngest.

Therefore, the second youngest person is Carlos.
</think>

To determine who should receive the senior discount, we need to identify the second youngest among Aisha, Ben, Carlos, and Diana.

**Given Information:**
1. **Aisha** is older than **Ben**.
2. **Ben** is older than **Carlos**.
3. **Carlos** is older than **Diana**.

**Step-by-Step Analysis:**

1. **Ordering the Ages:**
   - Aisha > Ben > Carlos > Diana

2. **Identifying the Second Youngest:**
   - **Aisha** (oldest)
   - **Ben** (second oldest)
   - **Carlos** (third oldest)
   - **Diana** (youngest)

### Prompt 5

#### Q. A disease affects 1 in 1000 people. A test for it is 99% accurate. You test positive. What is the probability you actually have the disease?

#### A. ~9%

In [19]:
ask(prompt5)

problem here about a disease and a test. Let me try to figure it out step by step. The disease affects 1 in 1000 people, which means the prevalence is 0.1%. Then there's a test that's 99% accurate. Hmm, I need to find the probability that if I test positive, I actually have the disease. That sounds like a classic probability question, maybe something to do with conditional probability.

Alright, let me recall. I think this is related to Bayes' theorem. Bayes' theorem helps us find the probability of an event given prior knowledge of related events. In this case, the event is having the disease, and the prior knowledge is the test result.

So, Bayes' theorem formula is:

P(Disease | Positive) = [P(Positive | Disease) * P(Disease)] / P(Positive)

I know that P(Disease) is 0.1%, or 0.001. The test is 99% accurate, so P(Positive | Disease) is 0.99. But wait, what about the false positive rate? If the test is 99% accurate, does that mean it's 99% accurate for both true positive and false po